<a href="https://colab.research.google.com/github/simsekergun/DeviceMetrology/blob/main/ZYe/DeviceMetrology_ZYE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## An Independent Study
Reference: https://doi.org/10.1364/PRJ.486379

Integrated dispersion data was shared at https://doi.org/10.5281/zenodo.7639970

Goal is predicting ring radius, width, and height from the integrated dispersion data!

In [1]:
# import every module we will or we might need
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from google.colab import files
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.models import Model
from keras.layers import Input
from keras.layers import Dropout
from keras.layers import concatenate
from tensorflow.keras import optimizers
from tensorflow.keras import backend
from keras.layers import LeakyReLU, PReLU
from tensorflow.keras.metrics import categorical_accuracy



In [2]:
XTrain = pd.read_csv('https://raw.githubusercontent.com/simsekergun/DeviceMetrology/refs/heads/main/ZYe/XTrain.csv',header=None)
YTrain = pd.read_csv('https://raw.githubusercontent.com/simsekergun/DeviceMetrology/refs/heads/main/ZYe/YTrain.csv',header=None)
XTest = pd.read_csv('https://raw.githubusercontent.com/simsekergun/DeviceMetrology/refs/heads/main/ZYe/XTest.csv',header=None)

In [3]:
XTrain.head(3)

,0,1,2,3,4,5,6,7,8,9,...,180,181,182,183,184,185,186,187,188,189
0,4.246400e+09,4.153000e+09,4.060900e+09,3.969700e+09,3.879600e+09,3.790400e+09,3.701900e+09,3.613900e+09,3.526700e+09,3.440500e+09,...,4.160900e+09,4.252700e+09,4.345500e+09,4.439400e+09,4.534600e+09,4.630700e+09,4.727600e+09,4.825400e+09,4.924000e+09,5.023800e+09
1,4.489500e+09,4.390700e+09,4.293100e+09,4.196600e+09,4.101100e+09,4.006600e+09,3.912800e+09,3.819600e+09,3.727300e+09,3.636200e+09,...,4.365400e+09,4.461500e+09,4.558800e+09,4.657400e+09,4.757500e+09,4.858700e+09,4.960900e+09,5.064000e+09,5.168200e+09,5.273700e+09
2,4.725500e+09,4.620900e+09,4.517000e+09,4.414000e+09,4.312100e+09,4.211700e+09,4.112700e+09,4.014600e+09,3.917400e+09,3.821400e+09,...,4.569200e+09,4.670800e+09,4.773400e+09,4.877100e+09,4.981900e+09,5.087700e+09,5.194400e+09,5.301800e+09,5.410000e+09,5.519200e+09


In [4]:
YTrain.head(3)

,0,1,2
0,225,1700,800
1,225,1700,810
2,225,1700,820


In [5]:
XTest.head(1)

,0,1,2,3,4,5,6,7,8,9,...,180,181,182,183,184,185,186,187,188,189
0,3.492200e+09,3.413400e+09,3.326200e+09,3.258800e+09,3.182500e+09,3.106700e+09,3.034300e+09,2.960400e+09,2.895000e+09,2.818700e+09,...,3.290000e+09,3.366000e+09,3.440600e+09,3.527500e+09,3.600200e+09,3.661400e+09,3.757700e+09,3.842200e+09,3.904900e+09,3.977000e+09


In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(XTrain)
X_test = scaler.fit_transform(XTest)

y_scaler = StandardScaler()
Y_train = y_scaler.fit_transform(YTrain)

In [7]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization, Activation, Input
from keras.optimizers import Adam
from keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping
from keras.regularizers import l1_l2
from keras.initializers import HeNormal

In [8]:
initializer = HeNormal()

model = Sequential()
model.add(Input(shape=(190,)))
model.add(Dense(500, kernel_initializer=initializer, activation='tanh'))
model.add(Dense(250, kernel_initializer=initializer, activation='tanh'))
model.add(Dropout(0.5))
model.add(Dense(100, kernel_initializer=initializer, activation='tanh'))
model.add(Dense(30, kernel_initializer=initializer, activation='tanh'))
model.add(BatchNormalization())
model.add(Dense(3))

opt = Adam(learning_rate=0.001, clipnorm=1.0)
model.compile(loss='huber', optimizer=opt, metrics=['mae'])

history = model.fit(
    X_train, Y_train,
    epochs=200,
    batch_size=32,
    verbose=0)

predictions = model.predict(X_test)
y_scaler.inverse_transform(predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step


array([[ 226.9255, 2001.406 ,  802.7887]], dtype=float32)

In [9]:
!pip install gpytorch

In [10]:
import torch
import gpytorch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

class FeatureExtractor(torch.nn.Module):
    def __init__(self, input_dim, embed_dim=10):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 500), torch.nn.ReLU(),
            torch.nn.Linear(500, 250), torch.nn.ReLU(),
            torch.nn.Linear(250, 100), torch.nn.ReLU(),
            torch.nn.Linear(100, embed_dim)
        )
    def forward(self, x):
        return self.net(x)

class GPRegressionModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood, feature_extractor):
        super().__init__(train_x, train_y, likelihood)
        self.feature_extractor = feature_extractor
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())
        self.scale_to_bounds = gpytorch.utils.grid.ScaleToBounds(-1., 1.)

    def forward(self, x):
        projected_x = self.feature_extractor(x)
        projected_x = self.scale_to_bounds(projected_x)
        mean_x = self.mean_module(projected_x)
        covar_x = self.covar_module(projected_x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


def train_dkl_model(X_train_t, y_train_t, embed_dim=10, n_iter=300, lr=0.01):
    feature_extractor = FeatureExtractor(X_train_t.shape[1], embed_dim).to(device)
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    model = GPRegressionModel(X_train_t, y_train_t, likelihood, feature_extractor).to(device)

    model.train()
    likelihood.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for i in range(n_iter):
        optimizer.zero_grad()
        output = model(X_train_t)
        loss = -mll(output, y_train_t)
        loss.backward()
        optimizer.step()
        if i % 25 == 0:
            print(f"Iter {i}/{n_iter} - Loss: {loss.item():.4f}")

    return model, likelihood


# Train one DKL model per output dimension
models = []
likelihoods = []
for i in range(Y_train_t.shape[1]):
    print(f"Training output {i}")
    m, l = train_dkl_model(X_train_t, Y_train_t[:, i], embed_dim=10, n_iter=300)
    models.append(m)
    likelihoods.append(l)

# Predictions
predictions = []
lower_bounds, upper_bounds = [], []
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    for m, l in zip(models, likelihoods):
        m.eval()
        l.eval()
        preds = l(m(X_test_t))
        predictions.append(preds.mean.cpu().numpy())
        lower, upper = preds.confidence_region()
        lower_bounds.append(lower.cpu().numpy())
        upper_bounds.append(upper.cpu().numpy())

predictions = np.stack(predictions, axis=1)  # shape (n_samples, 3)
predictions_original = y_scaler.inverse_transform(predictions)
print(predictions_original)

Training output 0
Iter 0/300 - Loss: 1.4692
Iter 25/300 - Loss: 1.3746
Iter 50/300 - Loss: 1.2139
Iter 75/300 - Loss: 1.0671
Iter 100/300 - Loss: 0.9679
Iter 125/300 - Loss: 0.8251
Iter 150/300 - Loss: 0.8095
Iter 175/300 - Loss: 0.7736
Iter 200/300 - Loss: 0.5967
Iter 225/300 - Loss: 0.5342
Iter 250/300 - Loss: 0.4879
Iter 275/300 - Loss: 0.4005
Training output 1
Iter 0/300 - Loss: 0.9930
Iter 25/300 - Loss: 0.8547
Iter 50/300 - Loss: 0.7271
Iter 75/300 - Loss: 0.5316
Iter 100/300 - Loss: 0.3745
Iter 125/300 - Loss: 0.2923
Iter 150/300 - Loss: 0.1239
Iter 175/300 - Loss: -0.0157
Iter 200/300 - Loss: -0.1513
Iter 225/300 - Loss: -0.2614
Iter 250/300 - Loss: -0.4073
Iter 275/300 - Loss: -0.5410
Training output 2
Iter 0/300 - Loss: 1.2982
Iter 25/300 - Loss: 0.9027
Iter 50/300 - Loss: 0.6591
Iter 75/300 - Loss: 0.5400
Iter 100/300 - Loss: 0.4005
Iter 125/300 - Loss: 0.2750
Iter 150/300 - Loss: 0.2634
Iter 175/300 - Loss: 0.0756
Iter 200/300 - Loss: -0.0513
Iter 225/300 - Loss: -0.1991
It